# 🏗️ 00 — SETU-Rail Setup: Catalog, Schemas, Volume

**Dhara module — Foundation cell**

Creates the Unity Catalog structure that every other notebook will use:
- Catalog: `setu_rail`
- Schemas: `bronze`, `silver`, `gold`
- Volume: `setu_rail.bronze.raw_files`

Then **directly imports every file from the Git-cloned `raw_data/` folder** into the Volume
using `dbutils.fs.cp` — no manual uploads needed.

**Run on:** Serverless compute. **Safe to re-run.**

---
### ✅ Pre-requisite
This repo must be cloned into your Databricks Workspace via **Repos** (Workspace → Repos → Add Repo).
All `raw_data/` files (CSVs, JSONs, PDFs) must be committed to the GitHub repo.
This notebook auto-detects the repo root and copies everything — zero manual steps.
---

## Step 1 — Create catalog

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS setu_rail
COMMENT 'SETU-Rail: AI copilot for Indian Railways (Bharat Bricks Hacks 2026, IIT Madras)';

USE CATALOG setu_rail;

## Step 2 — Create medallion schemas (Bronze / Silver / Gold)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS setu_rail.bronze
COMMENT 'Raw, append-only data — as delivered from sources';

CREATE SCHEMA IF NOT EXISTS setu_rail.silver
COMMENT 'Cleaned, joined, enriched data';

CREATE SCHEMA IF NOT EXISTS setu_rail.gold
COMMENT 'ML-ready and analytics-ready curated tables + views';

## Step 3 — Create Volume for raw files

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS setu_rail.bronze.raw_files
COMMENT 'Landing zone for raw CSVs and PDFs (train timings, air quality, Railway Rules, Railways Act)';

## Step 4 — Auto-detect repo root and copy all raw_data/ files into Volume

Uses `dbutils.fs.cp` (Databricks-native, fast) to copy directly from the
Workspace Git repo path into the Unity Catalog Volume.
No manual upload required. Re-runnable.

In [0]:
import os

# ── Detect this notebook's Workspace path ──────────────────────────────────
notebook_path = (
    dbutils.notebook.entry_point
    .getDbutils().notebook().getContext()
    .notebookPath().get()
)
# notebook_path is something like:
#   /Repos/<user>/setu-rail/01_dhara_data_engineering/00_setup_catalog_schema_volume
# We go two levels up to get the repo root.
repo_root_workspace = os.path.dirname(os.path.dirname(notebook_path))
raw_data_workspace  = repo_root_workspace + "/raw_data"

# Databricks Workspace paths are accessed via dbutils.fs with the /Workspace/ prefix
raw_data_dbfs       = "file:///Workspace" + raw_data_workspace
volume_path         = "/Volumes/setu_rail/bronze/raw_files"

print(f"📂 Repo root (Workspace): {repo_root_workspace}")
print(f"📂 raw_data dir:          {raw_data_workspace}")
print(f"📂 raw_data (file URI):   {raw_data_dbfs}")
print(f"📂 Volume target:         {volume_path}")
print()

# ── Verify the raw_data directory exists ──────────────────────────────────
try:
    files_in_raw = dbutils.fs.ls(raw_data_dbfs)
    print(f"Found {len(files_in_raw)} items in raw_data/:")
    for f in files_in_raw:
        print(f"  {f.name:<60} {f.size:>12,} bytes")
except Exception as e:
    raise FileNotFoundError(
        f"Cannot find raw_data/ at {raw_data_dbfs}.\n"
        f"Make sure:\n"
        f"  1. This repo is cloned via Workspace → Repos → Add Repo\n"
        f"  2. The raw_data/ folder exists in the GitHub repo with data committed\n"
        f"Original error: {e}"
    )

In [0]:
# ── Recursively copy every file from raw_data/ into the Volume ─────────────
# dbutils.fs.cp with recurse=True handles nested subdirs (data.gov/, kaggle_ds/)

copied = 0
skipped = 0
errors = []

def copy_dir(src_dir: str, dst_dir: str):
    """Recursively copy all files from src_dir (file:// URI) to dst_dir (Volume path)."""
    global copied, skipped
    try:
        items = dbutils.fs.ls(src_dir)
    except Exception as e:
        errors.append(f"Cannot list {src_dir}: {e}")
        return

    for item in items:
        src_path = item.path
        # item.name ends with '/' if it is a directory
        if item.name.endswith("/"):
            # Recurse into subdirectory — preserve folder structure in Volume
            sub_dir_name = item.name.rstrip("/")
            copy_dir(src_path, dst_dir + "/" + sub_dir_name)
        else:
            dst_path = dst_dir + "/" + item.name
            try:
                dbutils.fs.cp(src_path, dst_path, recurse=False)
                print(f"  ✅ {item.name:<55} ({item.size:>10,} bytes)")
                copied += 1
            except Exception as e:
                print(f"  ❌ {item.name}: {e}")
                errors.append(f"{item.name}: {e}")
                skipped += 1

print("Copying raw_data/ → Volume ...\n")
copy_dir(raw_data_dbfs, volume_path)

print(f"\n{'='*60}")
print(f"✅ Done.  Copied: {copied}   Skipped/errors: {skipped}")
if errors:
    print("\n⚠️  Errors:")
    for e in errors:
        print(f"   {e}")

## Step 5 — Verify Volume contents

In [0]:
def list_volume(path: str, indent: int = 0):
    """Recursively list Volume contents with sizes."""
    try:
        items = dbutils.fs.ls(path)
    except Exception:
        return
    for item in sorted(items, key=lambda x: x.name):
        prefix = "  " * indent
        if item.name.endswith("/"):
            print(f"{prefix}📁 {item.name}")
            list_volume(item.path, indent + 1)
        else:
            size_kb = item.size / 1024
            print(f"{prefix}📄 {item.name:<55} {size_kb:>8.1f} KB")

print("Volume contents:\n")
list_volume(volume_path)

## Step 6 — Validate expected files are present

In [0]:
# Collect all files in Volume into a flat list for validation
def collect_all_files(path: str) -> list:
    result = []
    try:
        items = dbutils.fs.ls(path)
    except Exception:
        return result
    for item in items:
        if item.name.endswith("/"):
            result.extend(collect_all_files(item.path))
        else:
            result.append(item.name)
    return result

all_files = collect_all_files(volume_path)
all_names = set(all_files)

# Define what we need for each pipeline stage
REQUIRED = {
    "Kaggle station graph (schedules.json)": "schedules.json",
    "Kaggle train metadata (trains.json)": "trains.json",
    "Kaggle station metadata (stations.json)": "stations.json",
    "Railway General Rules 1976 (PDF)": "1976_general_rules_railways_pdf.pdf",
    "Railways Act 1989 (PDF)": "the_railways_act__1989.pdf",
    "India Air Quality (CSV)": "india_air_quality.csv",
}

print("Validation:\n")
all_ok = True
for desc, fname in REQUIRED.items():
    found = fname in all_names
    status = "✅" if found else "❌ MISSING"
    print(f"  {status}  {desc}  ({fname})")
    if not found:
        all_ok = False

# Count Southern Railway CSVs
sr_csvs = [f for f in all_names if f.startswith("train_") and f.endswith(".csv")]
print(f"\n  ✅  Southern Railway CSVs found: {len(sr_csvs)} files")
for f in sorted(sr_csvs):
    print(f"       • {f}")

print()
if all_ok:
    print("🎉 All required files present. Pipeline is ready to run.")
    print("   Next step: open 01_ingest_all_sources.ipynb")
else:
    print("⚠️  Some files are missing. Check your raw_data/ folder in the GitHub repo.")

## Step 7 — Quick Delta smoke test
Write a tiny table to confirm Unity Catalog + Delta permissions are working before running the full pipeline.

In [0]:
%sql
-- Smoke test: write + read a Delta table in each schema
CREATE OR REPLACE TABLE setu_rail.bronze._setup_test (id INT, msg STRING)
USING DELTA
COMMENT 'Smoke test — safe to drop';

INSERT INTO setu_rail.bronze._setup_test VALUES (1, 'bronze ok');

CREATE OR REPLACE TABLE setu_rail.silver._setup_test (id INT, msg STRING) USING DELTA;
INSERT INTO setu_rail.silver._setup_test VALUES (1, 'silver ok');

CREATE OR REPLACE TABLE setu_rail.gold._setup_test   (id INT, msg STRING) USING DELTA;
INSERT INTO setu_rail.gold._setup_test   VALUES (1, 'gold ok');

SELECT 'bronze' AS schema, * FROM setu_rail.bronze._setup_test
UNION ALL
SELECT 'silver', * FROM setu_rail.silver._setup_test
UNION ALL
SELECT 'gold',   * FROM setu_rail.gold._setup_test;

In [0]:
%sql
-- Clean up smoke-test tables
DROP TABLE IF EXISTS setu_rail.bronze._setup_test;
DROP TABLE IF EXISTS setu_rail.silver._setup_test;
DROP TABLE IF EXISTS setu_rail.gold._setup_test;

In [0]:
# Final summary
print("""
╔══════════════════════════════════════════════════════════════╗
║           SETU-Rail Setup Complete ✅                       ║
╠══════════════════════════════════════════════════════════════╣
║  Catalog:   setu_rail                                       ║
║  Schemas:   bronze  /  silver  /  gold                      ║
║  Volume:    setu_rail.bronze.raw_files                      ║
║  Files:     Copied from GitHub repo raw_data/ → Volume      ║
╠══════════════════════════════════════════════════════════════╣
║  Run next:  01_ingest_all_sources.ipynb                     ║
╚══════════════════════════════════════════════════════════════╝
""")